# Time-evolved expectation values with `EvolveAndMeasure`

This notebook demonstrates how to:

1. Define a driven qubit Hamiltonian $H(t) = H_0 + f(t)\,H_1$
2. Build a Trotter evolution circuit via `EvolveAndMeasure`
3. Run **noiseless** and **noisy** simulations (QDK and Qiskit Aer backends)
4. Compare against a fine-grained matrix-exponential reference

In [ ]:
import numpy as np

from qdk_chemistry.algorithms import create
from qdk_chemistry.data import (
    AlgorithmRef, DrivenQubitHamiltonian, LatticeGraph, QuantumErrorProfile,
    QubitHamiltonian,
)
from qdk_chemistry.utils.model_hamiltonians import create_ising_hamiltonian

# Reduce logging output for demo
from qdk_chemistry.utils import Logger
Logger.set_global_level(Logger.LogLevel.off)


## Define the driven Hamiltonian

$$H(t) = H_0 + f(t)\,H_1$$

| Component | Expression | Role |
|-----------|-----------|------|
| $H_0$ | $J\,Z_0Z_1$ | Static Ising ZZ coupling |
| $H_1$ | $h\,(X_0+X_1)$ | Transverse field |
| $f(t)$ | $\sin(\omega\,t)$ | Smooth sinusoidal drive |

In [ ]:
lattice = LatticeGraph.chain(2)
h0 = create_ising_hamiltonian(lattice, j=1.0, h=0.0)   # ZZ coupling
h1 = create_ising_hamiltonian(lattice, j=0.0, h=0.5)   # transverse field

omega = 2 * np.pi
total_time = 2.0
dt = 0.1

td_hamiltonian = DrivenQubitHamiltonian(h0, h1, drive=lambda t: np.sin(omega * t))
observable = QubitHamiltonian(["ZZ"], np.array([1.0]))

In [ ]:
from qdk_chemistry.algorithms.state_preparation import identity_state_prep

state_prep = identity_state_prep(num_qubits=td_hamiltonian.num_qubits)

## Reference evolution (numerical exponentiation)

High-accuracy reference via fine-grained midpoint rule: each `dt` interval is subdivided into 100 substeps. This is not exact, but the error is negligible at this resolution.

In [ ]:
from scipy.sparse.linalg import expm_multiply

H0_sp, H1_sp = h0.to_matrix(sparse=True), h1.to_matrix(sparse=True)
drive = td_hamiltonian.drive

psi = np.zeros(2**h0.num_qubits, dtype=complex)
psi[0] = 1.0

n_substeps = 100
n_steps = max(1, round(total_time / dt))
dt_fine = dt / n_substeps

for step in range(n_steps):
    for sub in range(n_substeps):
        t = step * dt + (sub + 0.5) * dt_fine
        psi = expm_multiply(-1j * (H0_sp + drive(t) * H1_sp) * dt_fine, psi)

ZZ_diag = np.kron([1, -1], [1, -1]).astype(complex)
zz_ref = np.real(psi.conj() @ (ZZ_diag * psi))
print(f"Reference ⟨ZZ⟩: {zz_ref:.6f}")

## Noiseless Trotter simulation

Configure `EvolveAndMeasure`: the algorithm divides `[0, total_time]` into steps of size `dt`, evaluates $H(t)$ at each midpoint, and Trotterizes each step with `num_divisions` subdivisions.

In [ ]:
num_divisions = max(1, round(dt / 0.05))
shots = 100000

measure_simulation = create("measure_simulation", "evolve_and_measure")
measure_simulation.settings().set("evolution_builder", AlgorithmRef("hamiltonian_unitary_builder", "trotter", num_divisions=num_divisions, order=1))
measure_simulation.settings().set("circuit_executor", AlgorithmRef("circuit_executor", "qdk_full_state_simulator"))
measure_simulation.settings().set("total_time", total_time)
measure_simulation.settings().set("dt", dt)
measure_simulation.settings().set("shots", shots)

In [ ]:
results_noiseless = measure_simulation.run(td_hamiltonian, observables=[observable], state_prep=state_prep)
zz_noiseless = results_noiseless[0][0].energy_expectation_value
print(f"Noiseless ⟨ZZ⟩: {zz_noiseless:.6f}")

## Noisy simulations

Define a light depolarizing noise profile and run on both the QDK and Qiskit Aer backends.

In [ ]:
qdk_error_profile = QuantumErrorProfile(
    name="demo_noise",
    description="Light depolarizing noise on common gates",
    errors={
        "cx":  {"depolarizing_error": 0.001},
        "cz":  {"depolarizing_error": 0.001},
        "rzz": {"depolarizing_error": 0.001},
        "h":   {"depolarizing_error": 0.001},
        "rz":  {"depolarizing_error": 0.001},
        "rx":  {"depolarizing_error": 0.001},
        "s":   {"depolarizing_error": 0.001},
        "sdg": {"depolarizing_error": 0.001},
    },
)

In [ ]:
results_noisy_qdk = measure_simulation.run(td_hamiltonian, observables=[observable], state_prep=state_prep, noise=qdk_error_profile)
zz_noisy_qdk = results_noisy_qdk[0][0].energy_expectation_value
print(f"Noisy QDK ⟨ZZ⟩: {zz_noisy_qdk:.6f}")

In [ ]:
measure_aer = create("measure_simulation", "evolve_and_measure")
measure_aer.settings().set("evolution_builder", AlgorithmRef("hamiltonian_unitary_builder", "trotter", num_divisions=num_divisions, order=1))
measure_aer.settings().set("circuit_executor", AlgorithmRef("circuit_executor", "qiskit_aer_simulator"))
measure_aer.settings().set("total_time", total_time)
measure_aer.settings().set("dt", dt)
measure_aer.settings().set("shots", shots)

results_noisy_aer = measure_aer.run(td_hamiltonian, observables=[observable], state_prep=state_prep, noise=qdk_error_profile)
zz_noisy_aer = results_noisy_aer[0][0].energy_expectation_value
print(f"Noisy Aer ⟨ZZ⟩: {zz_noisy_aer:.6f}")

## Results

In [ ]:
print("Simulator               ⟨ZZ⟩")
print("─" * 36)
print(f"Reference                 {zz_ref: .6f}")
print(f"QDK  (noiseless)      {zz_noiseless: .6f}")
print(f"QDK  (noisy)          {zz_noisy_qdk: .6f}")
print(f"Aer  (noisy)          {zz_noisy_aer: .6f}")